<div style="background: linear-gradient(135deg, #0d1117 0%, #161b22 50%, #0d1117 100%); border: 2px solid #30363d; border-radius: 12px; padding: 30px; margin-bottom: 20px;">

<h1 style="color: #f0f6fc; text-align: center; font-size: 2em; margin: 0 0 12px 0;">
  🔒📊 프라이버시 점수 정량화 — SSIM · PSNR 기반 (ResNet50 실험용)
</h1>

<p style="color: #8b949e; text-align: center; margin: 0;">
블록 암호화 / 가우시안 블러 / 블록+블러 조합 세 가지를 한 번에 실행하여<br>
원본-변환 이미지쌍의 SSIM·PSNR로부터 Privacy Score를 계산하고 그래프 10개를 생성합니다.
</p>

</div>

## 📐 Privacy Score 공식 — 문헌 조사 결과 반영

**조사 결과**: Kiya 연구팀의 블록 스크램블링 계열 논문들, *P3: Toward
Privacy-Preserving Photo Sharing* (Ra et al., USENIX NSDI 2013,
arXiv:1302.5062), *PrivHAR-Bench* (2026), *GMFIM* (2022), *NegDL* (2021) 등
프라이버시 보호 이미지 연구 대부분은 SSIM과 PSNR을 **하나의 식으로 합치지 않고
독립적인 두 지표로 각각 보고**합니다 (둘 다 낮을수록 프라이버시 보호가 강함).
즉 "SSIM과 PSNR을 가중합해 하나의 점수로 만드는" 표준 공식은 문헌상 존재하지
않습니다.

대신, 원본과의 구조적 비유사도를 **하나의 정식 지표**로 나타내는 진짜 공식은
**DSSIM (Structural Dissimilarity Index)** 입니다. SSIM(Wang et al., 2004)을
거리로 변환한 것으로, *Unrecognizable Yet Identifiable* (Sarkar et al., 2024,
arXiv:2401.15048, Eq.14)에서 원본 대비 "인식 불가능성(unrecognizability)"을
정량화하는 데 그대로 사용된 공식입니다:

$$DSSIM(I, I') = \frac{1 - SSIM(I, I')}{2} \ \in [0, 1]$$

- $DSSIM = 0$ → 원본과 완전히 동일 (프라이버시 보호 없음)
- $DSSIM = 1$ → 원본과 구조적으로 완전히 반대 (최대 비유사도)

PSNR은 문헌 관행대로 **별도 지표**로 그대로 dB 단위 병기합니다. 참고 임계치
(P3 논문, arXiv:1302.5062):
- PSNR ≥ 35~40dB → 육안상 원본과 구분 불가능 ("perceptually lossless")
- PSNR ≈ 10~15dB → 실질적으로 원본 인식이 불가능한 수준으로 저하

`block_scramble` / `apply_gaussian_blur` 함수는 기존 정확도 실험 노트북과
100% 동일한 코드를 재사용합니다 (재현성 보장).

## 1️⃣ 라이브러리 임포트

In [1]:
# Cell 1: 라이브러리 임포트
import os, glob
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from PIL import Image, ImageFilter
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn

## 2️⃣ 한글 폰트 설정 (matplotlib 한글 완전 지원 + 다크테마)

In [2]:
# Cell 2: 한글 폰트 설정 (기존 노트북 Cell 2와 동일)
_CANDIDATES = [
    "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
    "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
    "/usr/share/fonts/truetype/unfonts-core/UnDotum.ttf",
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
]
font_path = next((p for p in _CANDIDATES if os.path.exists(p)), None)
if font_path is None:
    for p in fm.findSystemFonts(fontpaths=None, fontext="ttf"):
        if any(kw in p.lower() for kw in ["nanum", "malgun", "gulim", "undotum", "noto"]):
            font_path = p
            break

if font_path is None:
    print("⚠️  한글 폰트를 찾지 못했습니다. 기본 폰트 사용.")
    font_name = "DejaVu Sans"
else:
    cache_dir = matplotlib.get_cachedir()
    for f in glob.glob(os.path.join(cache_dir, "*.json")) + glob.glob(os.path.join(cache_dir, "*.cache")):
        try: os.remove(f)
        except: pass
    fm.fontManager.addfont(font_path)
    font_name = fm.FontProperties(fname=font_path).get_name()
    print(f"✅ 한글 폰트 적용: {font_name}  ({font_path})")

matplotlib.rcParams["font.family"]        = "sans-serif"
matplotlib.rcParams["font.sans-serif"]    = [font_name, "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
matplotlib.rcParams.update({
    "figure.facecolor":  "#0d1117",
    "axes.facecolor":    "#161b22",
    "axes.edgecolor":    "#444",
    "axes.labelcolor":   "#aaa",
    "xtick.color":       "#aaa",
    "ytick.color":       "#aaa",
    "grid.color":        "#333",
    "grid.linestyle":    "--",
    "grid.alpha":        0.5,
    "legend.facecolor":  "#222",
    "legend.labelcolor": "white",
    "text.color":        "white",
})
print("✅ 다크테마 rcParams 전역 적용 완료")

✅ 한글 폰트 적용: NanumGothic  (/usr/share/fonts/truetype/nanum/NanumGothic.ttf)
✅ 다크테마 rcParams 전역 적용 완료


## 3️⃣ 실험 설정

In [3]:
# Cell 3: 실험 설정 — 환경에 맞게 BASE_TRAIN, DESKTOP_DIR 확인
BASE_TRAIN   = "/workspace/Train"      # 원본(비암호화) 이미지 루트. 정확도 실험 노트북과 동일 경로
IMG_SIZE     = 512
SEED         = 42                       # block_scramble에 쓰인 것과 동일 시드
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")

BLOCK_SIZES  = [512, 256, 128, 64, 32, 16, 8, 4, 2, 1]
BLUR_SIGMAS  = [0, 1, 2, 4, 8, 16, 32, 64, 128, 256]

PSNR_MAX = 40.0   # dB, 참고용 정규화 상한 (DSSIM 계산에는 쓰이지 않음)
ALPHA    = 0.5    # 미사용 (DSSIM은 SSIM만 사용)

N_SAMPLES_PER_CLASS = None   # 속도 조절용. None이면 전체 이미지 사용

OUT_DIR   = "./privacy_score_outputs"
GRAPH_DIR = os.path.join(OUT_DIR, "eval_graphs_privacy_score")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(GRAPH_DIR, exist_ok=True)

# 바탕화면에 조건별 변환 예시 사진을 자동 저장 (도커 환경이면 실제 마운트된 바탕화면 경로로 수정)
DESKTOP_DIR    = os.path.expanduser("~/Desktop")
SAMPLE_IMG_DIR = os.path.join(DESKTOP_DIR, "privacy_sample_images")
for _sub in ("block", "blur", "combo"):
    os.makedirs(os.path.join(SAMPLE_IMG_DIR, _sub), exist_ok=True)

print(f"✅ 실험 설정 완료 | 블록 {len(BLOCK_SIZES)}개 · 블러 {len(BLUR_SIGMAS)}개 · 조합 {len(BLOCK_SIZES)*len(BLUR_SIGMAS)}개")
print(f"✅ 샘플 이미지 저장 경로: {SAMPLE_IMG_DIR}")

✅ 실험 설정 완료 | 블록 10개 · 블러 10개 · 조합 100개
✅ 샘플 이미지 저장 경로: /root/Desktop/privacy_sample_images


## 4️⃣ 변환 함수 (기존 정확도 실험 노트북과 완전히 동일)

In [4]:
# Cell 4: block_scramble / apply_gaussian_blur — 재현성을 위해 그대로 재사용
def block_scramble(img_array, seed, block_size):
    h, w, c = img_array.shape
    if block_size >= min(h, w):
        return img_array.copy()

    pad_h = (block_size - h % block_size) % block_size
    pad_w = (block_size - w % block_size) % block_size
    if pad_h > 0 or pad_w > 0:
        img_array = np.pad(img_array, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")

    ph, pw   = img_array.shape[:2]
    n_h, n_w = ph // block_size, pw // block_size
    blocks   = [img_array[i*block_size:(i+1)*block_size, j*block_size:(j+1)*block_size]
                for i in range(n_h) for j in range(n_w)]

    rng    = np.random.default_rng(seed)
    order  = rng.permutation(n_h * n_w)
    result = np.zeros_like(img_array)

    for new_idx, old_idx in enumerate(order):
        i, j = new_idx // n_w, new_idx % n_w
        result[i*block_size:(i+1)*block_size, j*block_size:(j+1)*block_size] = blocks[old_idx]

    return result[:h, :w]


def apply_gaussian_blur(img_array, sigma):
    if sigma == 0:
        return img_array.copy()
    pil_img = Image.fromarray(img_array.astype(np.uint8))
    blurred = pil_img.filter(ImageFilter.GaussianBlur(radius=float(sigma)))
    return np.array(blurred, dtype=np.uint8)

print("✅ block_scramble / apply_gaussian_blur 정의 완료")

✅ block_scramble / apply_gaussian_blur 정의 완료


## 5️⃣ 원본 이미지 로딩

In [5]:
# Cell 5: 원본 이미지 로딩
def list_class_dirs(directory):
    return sorted([d for d in os.listdir(directory) if os.path.isdir(os.path.join(directory, d))])

def list_image_files(directory):
    if not os.path.exists(directory):
        return []
    return sorted([f for f in os.listdir(directory) if f.lower().endswith(IMAGE_EXTENSIONS)])

def load_original_images(base_dir, n_per_class=None, seed=SEED):
    rng = np.random.default_rng(seed)
    images = []
    for cls in list_class_dirs(base_dir):
        files = list_image_files(os.path.join(base_dir, cls))
        if n_per_class is not None and len(files) > n_per_class:
            idx = rng.choice(len(files), size=n_per_class, replace=False)
            files = [files[i] for i in sorted(idx)]
        for f in files:
            p = os.path.join(base_dir, cls, f)
            try:
                img = Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
                images.append(np.array(img, dtype=np.uint8))
            except Exception as e:
                print(f"  ⚠️ 로드 실패: {p} ({e})")
    print(f"✅ 원본 이미지 {len(images)}장 로드 완료 (경로: {base_dir})")
    return images

images = load_original_images(BASE_TRAIN, n_per_class=N_SAMPLES_PER_CLASS, seed=SEED)
assert len(images) > 0, f"원본 이미지를 찾을 수 없습니다: {BASE_TRAIN}"

# 원본 샘플 1장을 바탕화면에 저장 (모든 조건의 변환 전/후 비교 기준)
Image.fromarray(images[0]).save(os.path.join(SAMPLE_IMG_DIR, "00_original_sample.png"))
print(f"✅ 원본 샘플 이미지 저장 → {SAMPLE_IMG_DIR}/00_original_sample.png")

✅ 원본 이미지 1097장 로드 완료 (경로: /workspace/Train)
✅ 원본 샘플 이미지 저장 → /root/Desktop/privacy_sample_images/00_original_sample.png


## 6️⃣ SSIM/PSNR + Privacy Score 계산 함수

In [6]:
# Cell 6: SSIM/PSNR 계산 + DSSIM 기반 Privacy Score
# DSSIM = (1-SSIM)/2  — Wang et al.(2004) SSIM을 거리로 변환한 정식 공식.
# Sarkar et al., 2024 "Unrecognizable Yet Identifiable"(arXiv:2401.15048, Eq.14)에서
# 원본 대비 인식 불가능성을 정량화하는 데 동일하게 사용됨.
def compute_ssim_psnr(orig, transformed):
    s = ssim_fn(orig, transformed, channel_axis=-1, data_range=255)
    try:
        p = psnr_fn(orig, transformed, data_range=255)
        if np.isinf(p):
            p = 100.0
    except Exception:
        p = 100.0
    return float(s), float(p)

def evaluate_condition(images, transform_fn, **kwargs):
    ssims, psnrs = [], []
    for img in images:
        t = transform_fn(img, **kwargs)
        s, p = compute_ssim_psnr(img, t)
        ssims.append(s)
        psnrs.append(p)
    return {
        "ssim_mean": float(np.mean(ssims)), "ssim_std": float(np.std(ssims)),
        "psnr_mean": float(np.mean(psnrs)), "psnr_std": float(np.std(psnrs)),
    }

def privacy_score_dssim(ssim_mean):
    """DSSIM = (1 - SSIM) / 2  (Wang et al. 2004 기반, 문헌에 실제 존재하는 공식)"""
    return float(np.clip((1.0 - ssim_mean) / 2.0, 0.0, 1.0))

print("✅ SSIM/PSNR 계산 + DSSIM 기반 Privacy Score 함수 정의 완료")

✅ SSIM/PSNR 계산 + DSSIM 기반 Privacy Score 함수 정의 완료


## 7️⃣ 🧩 블록 암호화 단독 실행 (10개)

In [7]:
# Cell 7: 블록 암호화 단독
rows = []
print("🧩 블록 암호화 단독 SSIM/PSNR 계산 중...")
for bs in BLOCK_SIZES:
    m = evaluate_condition(images, block_scramble, seed=SEED, block_size=bs)
    ps = privacy_score_dssim(m["ssim_mean"])
    rows.append({"condition": f"B{bs}_S0", "type": "block", "block_size": bs, "sigma": 0,
                 **m, "privacy_score": ps})
    print(f"  Block{bs:>4}: SSIM={m['ssim_mean']:.4f}  PSNR={m['psnr_mean']:6.2f}dB  DSSIM(PrivacyScore)={ps:.4f}")

    # 이 조건으로 변환한 샘플 사진 1장을 바탕화면에 저장
    sample = block_scramble(images[0], SEED, bs)
    Image.fromarray(sample).save(os.path.join(SAMPLE_IMG_DIR, "block", f"B{bs}_sample.png"))

print(f"✅ 블록 샘플 이미지 {len(BLOCK_SIZES)}장 저장 → {SAMPLE_IMG_DIR}/block/")

🧩 블록 암호화 단독 SSIM/PSNR 계산 중...


/usr/local/lib/python3.10/dist-packages/skimage/metrics/simple_metrics.py:168: RuntimeWarning: divide by zero encountered in scalar divide
  return 10 * np.log10((data_range**2) / err)


  Block 512: SSIM=1.0000  PSNR=100.00dB  DSSIM(PrivacyScore)=0.0000
  Block 256: SSIM=0.2407  PSNR=  6.03dB  DSSIM(PrivacyScore)=0.3796
  Block 128: SSIM=0.3183  PSNR=  7.15dB  DSSIM(PrivacyScore)=0.3408
  Block  64: SSIM=0.2969  PSNR=  7.38dB  DSSIM(PrivacyScore)=0.3516
  Block  32: SSIM=0.2752  PSNR=  7.78dB  DSSIM(PrivacyScore)=0.3624
  Block  16: SSIM=0.2115  PSNR=  7.51dB  DSSIM(PrivacyScore)=0.3943
  Block   8: SSIM=0.1166  PSNR=  7.50dB  DSSIM(PrivacyScore)=0.4417
  Block   4: SSIM=0.0358  PSNR=  7.51dB  DSSIM(PrivacyScore)=0.4821
  Block   2: SSIM=0.0095  PSNR=  7.50dB  DSSIM(PrivacyScore)=0.4952
  Block   1: SSIM=0.0078  PSNR=  7.50dB  DSSIM(PrivacyScore)=0.4961
✅ 블록 샘플 이미지 10장 저장 → /root/Desktop/privacy_sample_images/block/


## 8️⃣ 🌫️ 가우시안 블러 단독 실행 (10개)

In [8]:
# Cell 8: 가우시안 블러 단독
print("🌫️ 가우시안 블러 단독 SSIM/PSNR 계산 중...")
for sg in BLUR_SIGMAS:
    m = evaluate_condition(images, apply_gaussian_blur, sigma=sg)
    ps = privacy_score_dssim(m["ssim_mean"])
    rows.append({"condition": f"B512_S{sg}", "type": "blur", "block_size": 512, "sigma": sg,
                 **m, "privacy_score": ps})
    print(f"  Sigma{sg:>4}: SSIM={m['ssim_mean']:.4f}  PSNR={m['psnr_mean']:6.2f}dB  DSSIM(PrivacyScore)={ps:.4f}")

    # 이 조건으로 변환한 샘플 사진 1장을 바탕화면에 저장
    sample = apply_gaussian_blur(images[0], sg)
    Image.fromarray(sample).save(os.path.join(SAMPLE_IMG_DIR, "blur", f"S{sg}_sample.png"))

print(f"✅ 블러 샘플 이미지 {len(BLUR_SIGMAS)}장 저장 → {SAMPLE_IMG_DIR}/blur/")

🌫️ 가우시안 블러 단독 SSIM/PSNR 계산 중...
  Sigma   0: SSIM=1.0000  PSNR=100.00dB  DSSIM(PrivacyScore)=0.0000
  Sigma   1: SSIM=0.8973  PSNR= 32.67dB  DSSIM(PrivacyScore)=0.0513
  Sigma   2: SSIM=0.7748  PSNR= 26.21dB  DSSIM(PrivacyScore)=0.1126
  Sigma   4: SSIM=0.6578  PSNR= 22.22dB  DSSIM(PrivacyScore)=0.1711
  Sigma   8: SSIM=0.5958  PSNR= 19.62dB  DSSIM(PrivacyScore)=0.2021
  Sigma  16: SSIM=0.5761  PSNR= 17.34dB  DSSIM(PrivacyScore)=0.2120
  Sigma  32: SSIM=0.5635  PSNR= 14.99dB  DSSIM(PrivacyScore)=0.2182
  Sigma  64: SSIM=0.5427  PSNR= 13.10dB  DSSIM(PrivacyScore)=0.2286
  Sigma 128: SSIM=0.5145  PSNR= 11.60dB  DSSIM(PrivacyScore)=0.2428
  Sigma 256: SSIM=0.5033  PSNR= 10.14dB  DSSIM(PrivacyScore)=0.2484
✅ 블러 샘플 이미지 10장 저장 → /root/Desktop/privacy_sample_images/blur/


## 9️⃣ 🧩🌫️ 블록+블러 조합 실행 (100개)

In [9]:
# Cell 9: 블록+블러 조합 (10 x 10 = 100개)
# ※ 이전 버전에서는 조합 100개 각각의 SSIM/PSNR/PrivacyScore가 출력되지 않고
#    "완료" 메시지만 찍혔던 부분을 블록/블러와 동일하게 개별 결과가 보이도록 수정.
print("🧩🌫️ 블록+블러 조합 SSIM/PSNR 계산 중 (100개 조합)...")
for bs in BLOCK_SIZES:
    for sg in BLUR_SIGMAS:
        def combo_fn(img, block_size=bs, sigma=sg):
            return apply_gaussian_blur(block_scramble(img, SEED, block_size), sigma)
        m = evaluate_condition(images, combo_fn)
        ps = privacy_score_dssim(m["ssim_mean"])
        rows.append({"condition": f"B{bs}_S{sg}", "type": "combo", "block_size": bs, "sigma": sg,
                     **m, "privacy_score": ps})
        print(f"  B{bs:>4}_S{sg:<4}: SSIM={m['ssim_mean']:.4f}  PSNR={m['psnr_mean']:6.2f}dB  DSSIM(PrivacyScore)={ps:.4f}")

        # 이 조합 조건으로 변환한 샘플 사진 1장을 바탕화면에 저장
        sample = combo_fn(images[0])
        Image.fromarray(sample).save(os.path.join(SAMPLE_IMG_DIR, "combo", f"B{bs}_S{sg}_sample.png"))

print(f"✅ 조합 샘플 이미지 {len(BLOCK_SIZES) * len(BLUR_SIGMAS)}장 저장 → {SAMPLE_IMG_DIR}/combo/")
print(f"\n✅ 전체 {len(rows)}개 조건 계산 완료 (블록 10 + 블러 10 + 조합 100)")

🧩🌫️ 블록+블러 조합 SSIM/PSNR 계산 중 (100개 조합)...
  B 512_S0   : SSIM=1.0000  PSNR=100.00dB  DSSIM(PrivacyScore)=0.0000
  B 512_S1   : SSIM=0.8973  PSNR= 32.67dB  DSSIM(PrivacyScore)=0.0513
  B 512_S2   : SSIM=0.7748  PSNR= 26.21dB  DSSIM(PrivacyScore)=0.1126
  B 512_S4   : SSIM=0.6578  PSNR= 22.22dB  DSSIM(PrivacyScore)=0.1711
  B 512_S8   : SSIM=0.5958  PSNR= 19.62dB  DSSIM(PrivacyScore)=0.2021
  B 512_S16  : SSIM=0.5761  PSNR= 17.34dB  DSSIM(PrivacyScore)=0.2120
  B 512_S32  : SSIM=0.5635  PSNR= 14.99dB  DSSIM(PrivacyScore)=0.2182
  B 512_S64  : SSIM=0.5427  PSNR= 13.10dB  DSSIM(PrivacyScore)=0.2286
  B 512_S128 : SSIM=0.5145  PSNR= 11.60dB  DSSIM(PrivacyScore)=0.2428
  B 512_S256 : SSIM=0.5033  PSNR= 10.14dB  DSSIM(PrivacyScore)=0.2484
  B 256_S0   : SSIM=0.2407  PSNR=  6.03dB  DSSIM(PrivacyScore)=0.3796
  B 256_S1   : SSIM=0.2703  PSNR=  6.10dB  DSSIM(PrivacyScore)=0.3648
  B 256_S2   : SSIM=0.2873  PSNR=  6.16dB  DSSIM(PrivacyScore)=0.3563
  B 256_S4   : SSIM=0.2950  PSNR=  6.25dB  DSSIM(

## 🔟 결과 저장 (CSV / JSON)

In [10]:
# Cell 10: 결과 DataFrame 생성 및 저장
df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUT_DIR, "privacy_scores.csv"), index=False, encoding="utf-8-sig")
df.to_json(os.path.join(OUT_DIR, "privacy_scores.json"), orient="records", force_ascii=False, indent=2)
print(f"✅ 프라이버시 점수 저장 완료 → {OUT_DIR}/privacy_scores.csv / .json")
df.head(10)

✅ 프라이버시 점수 저장 완료 → ./privacy_score_outputs/privacy_scores.csv / .json


,condition,type,block_size,sigma,ssim_mean,ssim_std,psnr_mean,psnr_std,privacy_score
0,B512_S0,block,512,0,1.000000,0.000000,100.000000,0.000000,0.000000
1,B256_S0,block,256,0,0.240718,0.088481,6.031644,0.661237,0.379641
2,B128_S0,block,128,0,0.318333,0.085569,7.151123,0.508842,0.340834
3,B64_S0,block,64,0,0.296862,0.089389,7.378436,0.453124,0.351569
4,B32_S0,block,32,0,0.275218,0.089990,7.781824,0.516666,0.362391
5,B16_S0,block,16,0,0.211458,0.070218,7.505261,0.423014,0.394271
6,B8_S0,block,8,0,0.116586,0.046021,7.496825,0.444521,0.441707
7,B4_S0,block,4,0,0.035791,0.019122,7.508532,0.428272,0.482105
8,B2_S0,block,2,0,0.009533,0.002595,7.497547,0.429062,0.495233
9,B1_S0,block,1,0,0.007764,0.001136,7.502341,0.427474,0.496118


## 1️⃣1️⃣ 그래프 10개 생성

① 블록별 SSIM/PSNR  ② 블록별 Privacy Score  ③ 블러별 SSIM/PSNR  ④ 블러별 Privacy Score  
⑤ 블록 SSIM-vs-PSNR 산점도  ⑥ 블러 SSIM-vs-PSNR 산점도  ⑦ 조합 히트맵  
⑧ 유형별 분포 비교  ⑨ 점수 성분 분해  ⑩ 종합 대시보드

In [11]:
# Cell 11: 그래프 10개
ORANGE, CYAN, GREEN, RED, GRAY = "#ff9448", "#7dd3fc", "#6fd8a3", "#ff8a7a", "#8a99ac"

def _save(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(GRAPH_DIR, name), dpi=150)
    plt.show()

block_df = df[df["type"] == "block"].sort_values("block_size", ascending=False)
blur_df  = df[df["type"] == "blur"].sort_values("sigma")
combo_df = df[df["type"] == "combo"]

# ① 블록 크기별 SSIM/PSNR (이중 축)
fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()
ax1.plot(block_df["block_size"].astype(str), block_df["ssim_mean"], "o-", color=CYAN, label="SSIM")
ax2.plot(block_df["block_size"].astype(str), block_df["psnr_mean"], "s-", color=ORANGE, label="PSNR")
ax1.set_xlabel("블록 크기"); ax1.set_ylabel("SSIM", color=CYAN); ax2.set_ylabel("PSNR (dB)", color=ORANGE)
ax1.set_title("① 블록 크기별 SSIM / PSNR 변화")
ax1.invert_xaxis()
_save(fig, "01_block_ssim_psnr.png")

In [12]:
# ② 블록 크기별 Privacy Score
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(block_df["block_size"].astype(str), block_df["privacy_score"], color=GREEN)
ax.set_xlabel("블록 크기"); ax.set_ylabel("Privacy Score")
ax.set_title("② 블록 크기별 Privacy Score")
ax.invert_xaxis()
_save(fig, "02_block_privacy_score.png")

In [13]:
# ③ 블러 시그마별 SSIM/PSNR (이중 축)
fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()
ax1.plot(blur_df["sigma"].astype(str), blur_df["ssim_mean"], "o-", color=CYAN, label="SSIM")
ax2.plot(blur_df["sigma"].astype(str), blur_df["psnr_mean"], "s-", color=ORANGE, label="PSNR")
ax1.set_xlabel("블러 Sigma"); ax1.set_ylabel("SSIM", color=CYAN); ax2.set_ylabel("PSNR (dB)", color=ORANGE)
ax1.set_title("③ 블러 Sigma별 SSIM / PSNR 변화")
_save(fig, "03_blur_ssim_psnr.png")

In [14]:
# ④ 블러 시그마별 Privacy Score
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(blur_df["sigma"].astype(str), blur_df["privacy_score"], color=GREEN)
ax.set_xlabel("블러 Sigma"); ax.set_ylabel("Privacy Score")
ax.set_title("④ 블러 Sigma별 Privacy Score")
_save(fig, "04_blur_privacy_score.png")

In [15]:
# ⑤ 블록 단독: SSIM vs PSNR 산점도 (색=block size)
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(block_df["ssim_mean"], block_df["psnr_mean"], c=block_df["block_size"],
                 cmap="cool", s=90, edgecolors="white")
for _, r in block_df.iterrows():
    ax.annotate(str(r["block_size"]), (r["ssim_mean"], r["psnr_mean"]), fontsize=8, color="white")
fig.colorbar(sc, ax=ax, label="블록 크기")
ax.set_xlabel("SSIM"); ax.set_ylabel("PSNR (dB)")
ax.set_title("⑤ 블록 단독 — SSIM vs PSNR")
_save(fig, "05_block_ssim_vs_psnr.png")

In [16]:
# ⑥ 블러 단독: SSIM vs PSNR 산점도 (색=sigma)
fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(blur_df["ssim_mean"], blur_df["psnr_mean"], c=blur_df["sigma"],
                 cmap="autumn", s=90, edgecolors="white")
for _, r in blur_df.iterrows():
    ax.annotate(str(r["sigma"]), (r["ssim_mean"], r["psnr_mean"]), fontsize=8, color="white")
fig.colorbar(sc, ax=ax, label="Sigma")
ax.set_xlabel("SSIM"); ax.set_ylabel("PSNR (dB)")
ax.set_title("⑥ 블러 단독 — SSIM vs PSNR")
_save(fig, "06_blur_ssim_vs_psnr.png")

In [17]:
# ⑦ 블록×블러 조합 Privacy Score 히트맵
pivot = combo_df.pivot(index="block_size", columns="sigma", values="privacy_score")
pivot = pivot.reindex(index=BLOCK_SIZES[::-1], columns=BLUR_SIGMAS)
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(pivot.values, cmap="magma", aspect="auto")
ax.set_xticks(range(len(BLUR_SIGMAS))); ax.set_xticklabels(BLUR_SIGMAS)
ax.set_yticks(range(len(BLOCK_SIZES))); ax.set_yticklabels(BLOCK_SIZES[::-1])
ax.set_xlabel("블러 Sigma"); ax.set_ylabel("블록 크기")
ax.set_title("⑦ 블록×블러 조합 Privacy Score 히트맵")
fig.colorbar(im, ax=ax, label="Privacy Score")
_save(fig, "07_combo_heatmap.png")

In [18]:
# ⑧ 블록 단독 vs 블러 단독 vs 조합 Privacy Score 분포 비교
fig, ax = plt.subplots(figsize=(7, 5))
data = [block_df["privacy_score"], blur_df["privacy_score"], combo_df["privacy_score"]]
bp = ax.boxplot(data, labels=["블록 단독", "블러 단독", "블록+블러 조합"], patch_artist=True)
for patch, c in zip(bp["boxes"], [CYAN, ORANGE, GREEN]):
    patch.set_facecolor(c); patch.set_alpha(0.6)
ax.set_ylabel("Privacy Score")
ax.set_title("⑧ 변환 유형별 Privacy Score 분포 비교")
_save(fig, "08_type_comparison_boxplot.png")

/tmp/ipykernel_96419/3856397871.py:4: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=["블록 단독", "블러 단독", "블록+블러 조합"], patch_artist=True)


In [19]:
# ⑨ DSSIM(Privacy Score) vs PSNR 비교 — 블록/블러 단독 조건 전체 (참고 임계치 포함)
# 임계치 출처: P3: Toward Privacy-Preserving Photo Sharing (Ra et al., 2013, arXiv:1302.5062)
top = pd.concat([block_df, blur_df])
fig, ax1 = plt.subplots(figsize=(10, 6))
ax2 = ax1.twinx()
ax1.bar(top["condition"], top["privacy_score"], color=GREEN, alpha=0.7, label="DSSIM (Privacy Score)")
ax2.plot(top["condition"], top["psnr_mean"], "o-", color=ORANGE, label="PSNR (dB)")
ax2.axhline(40, color=RED, linestyle="--", linewidth=1, label="40dB: perceptually lossless (P3, 2013)")
ax2.axhline(15, color=CYAN, linestyle="--", linewidth=1, label="15dB: practically useless (P3, 2013)")
ax1.set_ylabel("DSSIM (Privacy Score)", color=GREEN); ax2.set_ylabel("PSNR (dB)", color=ORANGE)
ax1.set_title("⑨ DSSIM(Privacy Score) vs PSNR 비교 (문헌 임계치 포함)")
plt.setp(ax1.get_xticklabels(), rotation=60, ha="right", fontsize=7)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=7)
_save(fig, "09_dssim_vs_psnr.png")

In [20]:
# ⑩ 종합 대시보드
fig = plt.figure(figsize=(14, 9))
gs = GridSpec(2, 3, figure=fig)

ax = fig.add_subplot(gs[0, 0])
ax.plot(block_df["block_size"].astype(str), block_df["privacy_score"], "o-", color=GREEN)
ax.set_title("블록별 Privacy Score"); ax.invert_xaxis()

ax = fig.add_subplot(gs[0, 1])
ax.plot(blur_df["sigma"].astype(str), blur_df["privacy_score"], "o-", color=GREEN)
ax.set_title("Sigma별 Privacy Score")

ax = fig.add_subplot(gs[0, 2])
im = ax.imshow(pivot.values, cmap="magma", aspect="auto")
ax.set_title("조합 히트맵"); ax.set_xticks([]); ax.set_yticks([])

ax = fig.add_subplot(gs[1, 0])
ax.scatter(block_df["ssim_mean"], block_df["psnr_mean"], c=CYAN, s=50)
ax.set_title("블록: SSIM vs PSNR"); ax.set_xlabel("SSIM"); ax.set_ylabel("PSNR")

ax = fig.add_subplot(gs[1, 1])
ax.scatter(blur_df["ssim_mean"], blur_df["psnr_mean"], c=ORANGE, s=50)
ax.set_title("블러: SSIM vs PSNR"); ax.set_xlabel("SSIM"); ax.set_ylabel("PSNR")

ax = fig.add_subplot(gs[1, 2])
ax.hist(combo_df["privacy_score"], bins=15, color=GRAY)
ax.set_title("조합 100개 Privacy Score 분포")

fig.suptitle("⑩ 프라이버시 점수(DSSIM) 종합 대시보드", fontsize=15, color="white")
_save(fig, "10_dashboard.png")

print(f"\n✅ 그래프 10개 저장 완료 → {GRAPH_DIR}/")


✅ 그래프 10개 저장 완료 → ./privacy_score_outputs/eval_graphs_privacy_score/
